# Aula 1 — Texto como dado

**Do conteúdo humano ao corpus analisável**

Textos parecem naturais para pessoas, mas computadores não trabalham com significado da mesma forma que nós.

Antes de representar palavras numericamente ou treinar modelos, precisamos dar um passo fundamental: **organizar texto como dado observável, inspecionável e reproduzível**.

Nesta aula, você vai aprender a enxergar uma coleção de textos como um pequeno sistema de dados — com documentos, campos, metadados, qualidade e contexto.


## 1. Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

- explicar o que significa tratar texto como dado;
- distinguir conteúdo textual de metadados;
- reconhecer documento, corpus e rótulo;
- inspecionar uma pequena coleção de textos com Python;
- identificar problemas básicos de qualidade, como campos vazios e duplicatas;
- compreender por que essa etapa vem antes de qualquer técnica de NLP.


## Antes de começar — trabalhe na sua própria cópia

Se você abriu o notebook oficial do TIL, crie sua cópia no Kaggle antes de executar qualquer célula.

```text
Notebook oficial = referência do curso
Cópia do aluno    = ambiente pessoal de aprendizagem
```


## 2. Por que texto precisa virar dado?

Imagine milhares de mensagens de clientes, contratos, e-mails, respostas de pesquisa ou documentos administrativos.

Para uma pessoa, cada item possui significado. Para um sistema analítico, porém, precisamos primeiro responder perguntas mais básicas:

- quantos documentos existem?
- quais campos descrevem cada documento?
- existem textos vazios?
- há duplicatas?
- existem categorias ou rótulos conhecidos?
- há metadados importantes, como canal, data ou origem?

Esse trabalho parece simples, mas é a fundação de praticamente todo projeto sério de Text Intelligence.

**Por que isso importa?** Porque modelos sofisticados não corrigem automaticamente dados mal definidos, incompletos ou inconsistentes.

> 📘 **Glossário:** [dado](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#data), [dado observável](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#observable-data), [dado inspecionável](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#inspectable-data) e [reprodutibilidade](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#reproducibility).


## 3. Três conceitos fundamentais

### Documento

Uma unidade individual de texto que será analisada.

Exemplos: uma mensagem, um e-mail, uma avaliação, uma notícia, uma página ou um contrato.

### Corpus

Uma coleção de documentos organizada para análise.

### Rótulo (*label*)

Uma categoria conhecida associada a um documento.

Por exemplo, uma mensagem pode ter o rótulo `reclamacao`, `elogio` ou `duvida`.

> Nem todo corpus possui rótulos. Quando possui, eles podem ser usados mais tarde em tarefas supervisionadas de Machine Learning.

> 📘 **Glossário:** [documento](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#document), [corpus](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#corpus), [rótulo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#label--rótulo) e [classe](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#class).


## 4. Nosso primeiro corpus

Vamos trabalhar com uma coleção mínima e artificial de mensagens de atendimento.

O objetivo **não é fazer NLP ainda**. Vamos apenas observar como texto pode ser organizado junto com metadados.

Execute a próxima célula. Ela cria seis registros e os transforma em uma tabela usando `pandas`.


In [ ]:
import pandas as pd

records = [
    {"id": 1, "channel": "chat", "label": "duvida", "text": "Como altero minha senha?"},
    {"id": 2, "channel": "email", "label": "reclamacao", "text": "Meu pedido ainda não chegou."},
    {"id": 3, "channel": "chat", "label": "elogio", "text": "O atendimento foi excelente!"},
    {"id": 4, "channel": "chat", "label": "duvida", "text": "Posso pagar a fatura amanhã?"},
    {"id": 5, "channel": "email", "label": "reclamacao", "text": "Meu pedido ainda não chegou."},
    {"id": 6, "channel": "chat", "label": None, "text": ""},
]

df = pd.DataFrame(records)
df


### O que observar

Cada linha representa um **documento**.

A coluna `text` contém o conteúdo textual. As demais colunas funcionam como metadados:

- `id`: identificador;
- `channel`: origem da mensagem;
- `label`: categoria conhecida;
- `text`: conteúdo que será analisado.

**Checkpoint 1:** confirme que você consegue separar mentalmente **conteúdo** de **metadados**.

> 📘 **Glossário:** [dataset](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#dataset) e [metadado](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#metadata).


## 5. Inspeção inicial do corpus

Antes de qualquer transformação textual, devemos conhecer a coleção.

Execute a célula abaixo. Ela calcula apenas indicadores básicos de estrutura e qualidade.


In [ ]:
summary = {
    "documents": len(df),
    "columns": list(df.columns),
    "empty_texts": int(df["text"].str.strip().eq("").sum()),
    "missing_labels": int(df["label"].isna().sum()),
    "duplicated_texts": int(df["text"].duplicated().sum()),
    "channels": df["channel"].value_counts().to_dict(),
    "labels": df["label"].value_counts(dropna=False).to_dict(),
}

summary


### O que interpretar

Esse resumo já revela problemas importantes:

- existe pelo menos um texto vazio;
- existe pelo menos um rótulo ausente;
- existe texto duplicado;
- os documentos vêm de mais de um canal;
- as categorias não aparecem na mesma frequência.

Nenhum modelo foi treinado e nenhuma técnica de NLP foi aplicada. Mesmo assim, já aprendemos algo essencial sobre os dados.

**Checkpoint 2:** antes de pensar em algoritmo, pergunte sempre: *o que exatamente tenho em mãos?*

> 📘 **Glossário:** [texto vazio](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#empty-text), [valor ausente](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#missing-value) e [duplicata](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#duplicate).


## 6. Texto também possui propriedades mensuráveis

Ainda sem interpretar palavras, podemos medir propriedades simples dos documentos.

A próxima célula adiciona o número de caracteres de cada texto.


In [ ]:
df["n_chars"] = df["text"].str.len()
df[["id", "label", "text", "n_chars"]]


### Por que isso importa?

Comprimento não representa significado, mas pode revelar anomalias, diferenças de comportamento e problemas de qualidade.

Mais adiante, aprenderemos representações muito mais informativas. Por enquanto, basta perceber que **texto pode ser observado tanto como linguagem quanto como objeto de dados**.


## 7. Exercício guiado

Agora você vai responder às perguntas **com código Python executável**.

Use o `DataFrame df` e, quando for útil, o dicionário `summary` criado anteriormente.

Seu código deve produzir respostas para:

1. Quantos documentos existem no corpus?
2. Quantos textos estão vazios?
3. Quantos rótulos estão ausentes?
4. Existe algum texto duplicado?
5. Qual coluna representa o conteúdo textual?
6. Cite dois exemplos de metadados presentes no corpus.

Tente construir a solução antes de consultar a dica.


In [ ]:
# Escreva sua solução aqui.

# Exemplo de formato esperado:
# print("Documentos:", ...)
# print("Textos vazios:", ...)
# print("Rótulos ausentes:", ...)
# print("Há duplicatas?", ...)
# print("Coluna de texto:", ...)
# print("Exemplos de metadados:", ...)


### Dica e solução

Execute **uma vez** a próxima célula para preparar `q1.hint()` e `q1.solution()`.

A dica não entrega a resposta pronta: ela indica quais recursos do Python e do `pandas` podem ajudar.


In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução\n\n{self._solution_text}"))

q1 = TILExercise(
    hint_text=(
        "Use **Python + pandas**. Você já possui o `DataFrame df` e o dicionário `summary`. "
        "Alguns recursos úteis são: `len(df)`, `Series.str.strip()`, `Series.eq()`, "
        "`Series.isna()`, `Series.duplicated()`, `df.columns` e acesso a chaves de dicionário como `summary['documents']`.\n\n"
        "Você também pode reutilizar resultados que já foram calculados no dicionário `summary`. "
        "Por exemplo:\n\n"
        "```python\n"
        "print('Documentos:', summary['documents'])\n"
        "print('Textos vazios:', summary['empty_texts'])\n"
        "```\n\n"
        "Isso mostra duas estratégias válidas: calcular diretamente a partir de `df` ou reutilizar um resultado intermediário já armazenado."
    ),
    solution_text=(
        "Uma possível solução executável é:\n\n"
        "```python\n"
        "print('Documentos:', len(df))\n"
        "print('Textos vazios:', df['text'].str.strip().eq('').sum())\n"
        "print('Rótulos ausentes:', df['label'].isna().sum())\n"
        "print('Há duplicatas?', df['text'].duplicated().any())\n"
        "print('Coluna de texto:', 'text')\n"
        "print('Exemplos de metadados:', ['id', 'channel'])\n"
        "```\n\n"
        "Uma alternativa é reutilizar valores já armazenados em `summary`, por exemplo:\n\n"
        "```python\n"
        "print('Documentos:', summary['documents'])\n"
        "print('Textos vazios:', summary['empty_texts'])\n"
        "print('Rótulos ausentes:', summary['missing_labels'])\n"
        "```"
    ),
)

print("Exercício preparado. Tente resolver antes de usar q1.hint() ou q1.solution().")


In [ ]:
# Remova o # da linha abaixo se quiser uma dica.
# q1.hint()


In [ ]:
# Remova o # da linha abaixo para revelar a resposta.
# q1.solution()


## 8. Reprodutibilidade

Esta aula foi desenhada para executar sem datasets externos.

Configuração esperada:

- linguagem: Python;
- biblioteca: `pandas`;
- acelerador: CPU;
- internet: desabilitada;
- dataset: criado dentro do próprio notebook.

Isso permite que qualquer aluno reproduza exatamente o mesmo corpus inicial.


## 9. Resumo

Nesta aula, você aprendeu que antes de fazer NLP precisamos definir e observar os dados textuais.

Você viu que:

- um **documento** é uma unidade de texto;
- um **corpus** é uma coleção de documentos;
- um **rótulo** é uma categoria conhecida associada a um documento;
- conteúdo textual e metadados cumprem papéis diferentes;
- problemas de qualidade podem ser detectados antes de qualquer modelagem;
- textos também possuem propriedades mensuráveis.

### Ideia principal

```text
Antes de representar texto,
precisamos saber exatamente
que coleção de textos estamos representando.
```

**Fim da Aula 1.**
